# PPMI Subgroup Metadata Preparation

This notebook builds `subgroups.csv`, the per-patient-per-visit metadata
table (subgroup label, age at visit, gender) that other PPMI notebooks
(e.g. the MDS-UPDRS prep notebook) join against.

**What this notebook does:**
1. Loads the idiopathic-PD subgroup assignments and the PPMI demographic
   linkage table, and merges them.
2. Expands the data to a full patient x visit grid (one row per patient
   per visit in the visit schedule, even for visits with no recorded
   demographic row).
3. Fills in `GENDER` and `Group` for the newly-added grid rows (they're
   constant per patient, so this is a same-patient carry-fill rather than
   a real imputation).
4. Fills in `AGE_AT_VISIT` for the newly-added grid rows by incrementing
   0.5 years (the visit spacing) from the previous known age.
5. Relabels the subgroup codes and writes the result to
   `data/02_processed/PPMI/subgroups.csv`.


In [61]:
import pandas as pd
import numpy as np
from scipy import stats

### Load Subgroup Labels and Demographic Linkage

In [62]:
# Idiopathic-PD subgroup assignment per patient (data-driven subgroup,
# still using the raw 0/1/2 coding at this point — relabeled later).
subgroup = pd.read_csv('./../../data/01_raw/PPMI/idiopathic_subgroup.csv', index_col=0)
subgroup.columns = ['PATNO', 'Group']

In [17]:
remove = [3005,
3062,
3133,
3329,
3527,
3559,
3583,
3589,
3413,
]

In [18]:
subgroup = subgroup[~subgroup['PATNO'].isin(remove)]

In [63]:
subgroup

,PATNO,Group
0,3001,0
1,3002,2
2,3003,2
3,3005,2
4,3006,0
...,...,...
538,149945,1
539,150414,2
540,152582,0
541,153194,2


In [31]:
subgroup = subgroup.iloc[:-147 , :]

In [64]:
# PPMI's master linkage table: one row per patient-visit, with
# demographic fields such as AGE_AT_VISIT and GENDER.
meta = pd.read_csv('./../../data/01_raw/PPMI/PPMI_META_LINKAGE.csv')

In [65]:
import pickle

subgroups = pd.read_csv('./../../data/02_processed/PPMI/subgroups.csv', index_col=0)['PATNO']

with open('./../../data/02_processed/PPMI/P1_MDSUPDRS.pkl' , 'rb') as file : 
    p1 = pickle.load(file)

with open('./../../data/02_processed/PPMI/P2_MDSUPDRS.pkl' , 'rb') as file : 
    p2 = pickle.load(file)

with open('./../../data/02_processed/PPMI/P3ON_MDSUPDRS.pkl' , 'rb') as file : 
    p3_on = pickle.load(file)

with open('./../../data/02_processed/PPMI/P3OFF_MDSUPDRS.pkl' , 'rb') as file : 
    p3_off = pickle.load(file)

Quick sanity check on the raw linkage table before merging:

In [66]:
meta

,PATNO,ENROLL_DATE,ENROLL_AGE,CONCOHORT_DEFINITION,Subgroup,GENDER,EVENT_ID,EDUCYRS,AGE_AT_VISIT,BIRTHDT,race,ageonset,agediag,primdiag,changedx
0,40932,01/05/2014,62.0,Parkinson's Disease,Genetic,0.0,BL,18.0,62.0,NaN,NaN,NaN,NaN,NaN,NaN
1,40989,01/05/2014,88.0,Parkinson's Disease,Genetic,0.0,BL,12.0,88.0,NaN,NaN,NaN,NaN,NaN,NaN
2,41821,01/02/2017,87.0,Parkinson's Disease,Genetic,0.0,BL,14.0,87.0,NaN,NaN,NaN,NaN,NaN,NaN
3,50770,01/02/2015,64.0,Prodromal,Genetic,0.0,BL,17.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN
4,50770,01/02/2015,64.0,Prodromal,Genetic,0.0,V06,17.0,66.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7975,203816,01/02/2023,71.9,Prodromal,NaN,1.0,BL,NaN,71.9,01/03/1951,NaN,NaN,NaN,NaN,NaN
7976,204012,01/03/2023,73.0,Parkinson's Disease,NaN,1.0,BL,NaN,73.0,01/03/1950,NaN,NaN,NaN,NaN,NaN
7977,205186,NaN,NaN,Parkinson's Disease,NaN,1.0,BL,NaN,71.2,01/12/1951,NaN,NaN,NaN,NaN,NaN
7978,211902,NaN,NaN,Parkinson's Disease,NaN,1.0,BL,NaN,64.6,01/07/1958,NaN,NaN,NaN,NaN,NaN


In [67]:
# Attach the subgroup label to every patient-visit row, keeping only the
# columns needed downstream.
subgroup_meta = pd.merge(meta, subgroup, on='PATNO')[
    ['PATNO', 'AGE_AT_VISIT', 'EVENT_ID', 'GENDER', 'Group', 'agediag']
].drop_duplicates()

Quick sanity check on the merged subgroup metadata:

In [68]:
p3_off.columns

Index(['PATNO', 'EVENT_ID', 'NHY', 'NP3SPCH', 'NP3FACXP', 'NP3RIGN',
       'NP3RIGRU', 'NP3RIGLU', 'NP3RIGRL', 'NP3RIGLL', 'NP3FTAPR', 'NP3FTAPL',
       'NP3HMOVR', 'NP3HMOVL', 'NP3PRSPR', 'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL',
       'NP3LGAGR', 'NP3LGAGL', 'NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL',
       'NP3POSTR', 'NP3BRADY', 'NP3PTRMR', 'NP3PTRML', 'NP3KTRMR', 'NP3KTRML',
       'NP3RTARU', 'NP3RTALU', 'NP3RTARL', 'NP3RTALL', 'NP3RTALJ', 'NP3RTCON',
       'NP3TOT'],
      dtype='object')

In [69]:
mds = pd.merge(pd.merge(p1, p2 , on=['PATNO', 'EVENT_ID']) , p3_off , on=['PATNO', 'EVENT_ID'])

In [70]:
mds['TOT'] = mds['NP1TOT'] + mds['NP2PTOT'] + mds['NP3TOT']

In [71]:
p3_off

,PATNO,EVENT_ID,NHY,NP3SPCH,NP3FACXP,NP3RIGN,NP3RIGRU,NP3RIGLU,NP3RIGRL,NP3RIGLL,...,NP3PTRML,NP3KTRMR,NP3KTRML,NP3RTARU,NP3RTALU,NP3RTARL,NP3RTALL,NP3RTALJ,NP3RTCON,NP3TOT
9,3001,BL,1.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0
12,3001,V02,2.0,1.0,1.0,0.0,2.0,1.0,1.0,0.0,...,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0
14,3001,V04,2.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0
15,3001,V05,2.0,2.0,2.0,0.0,3.0,1.0,1.0,1.0,...,1.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0,29.0
16,3001,V06,2.0,2.0,2.0,1.0,3.0,2.0,2.0,2.0,...,1.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,2.0,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28878,157424,BL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
28881,157424,V04,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
28882,157424,V06,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
28883,157424,V08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [72]:
tst = pd.merge(subgroup[['PATNO' , 'Group']].drop_duplicates() , p3_off[['PATNO' , 'EVENT_ID','NHY']] , on = 'PATNO')
tst = tst[tst['EVENT_ID'] == 'BL']
tst

,PATNO,Group,EVENT_ID,NHY
0,3001,0,BL,1.0
13,3002,2,BL,2.0
23,3003,2,BL,2.0
35,3006,0,BL,2.0
38,3007,2,BL,2.0
...,...,...,...,...
4354,149945,1,BL,2.0
4361,150414,2,BL,2.0
4366,152582,0,BL,2.0
4373,153194,2,BL,0.0


In [73]:
tst.groupby('Group').count()

,PATNO,EVENT_ID,NHY
Group,,,
0,110,110,110
1,265,265,265
2,159,159,159


In [74]:


contingency = pd.crosstab(subgroup_meta.drop_duplicates('PATNO')['Group'], 
                            subgroup_meta.drop_duplicates('PATNO')['GENDER'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square: χ²={chi2:.3f}, p={p_val:.4f}")

Chi-square: χ²=0.323, p=0.8508


In [75]:
from scipy import stats

df = tst.drop_duplicates('PATNO')

# One-way ANOVA (parametric) - assumes normality, equal variance
groups = [df.loc[df['Group'] == g, 'NHY'].dropna() 
          for g in df['Group'].unique()]
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.4f}")

# Kruskal-Wallis (non-parametric) - safer default for clinical scores,
# especially ordinal ones like Hoehn & Yahr, or skewed ones like UPDRS
h_stat, p_val = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H={h_stat:.3f}, p={p_val:.4f}")

ANOVA: F=0.233, p=0.7925
Kruskal-Wallis: H=0.938, p=0.6257


### Define the Visit Schedule

Same visit schedule used in the MDS-UPDRS prep notebook — restricted to the visits for which genomic data is available.

In [76]:
# Visit codes used throughout this notebook.
events = ['BL', 'V02', 'V04', 'V05',
          'V06', 'V07', 'V08', 'V09', 'V10',
          'V11', 'V12', 'V13', 'V14', 'V15',
          'V16', 'V17', 'V18', 'V19', 'V20']

# Full PPMI visit vocabulary, kept for reference/lookup.
all_events = ['SC', 'BL', 'V01', 'V02', 'V03', 'V04', 'V05',
              'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12',
              'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
              'ST', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25']

# Maps each visit to an approximate elapsed time in years (visits are
# ~6 months apart). Not used further in this notebook, kept for reference.
x_list = pd.DataFrame(np.arange(0, 9.5, 0.5), index=events, columns=['response'])

### Build a Full Patient x Visit Grid

Ensures every patient has exactly one row per visit in `events`, regardless of which visits actually have a demographic record.

In [77]:
events_df = pd.DataFrame()
for patno in subgroup_meta['PATNO'].unique():
    df_tmp = pd.DataFrame({'PATNO': [patno for i in range(len(events))], 'EVENT_ID': events})
    events_df = pd.concat([events_df, df_tmp])

In [78]:
events_df

,PATNO,EVENT_ID
0,3001,BL
1,3001,V02
2,3001,V04
3,3001,V05
4,3001,V06
...,...,...
14,157424,V16
15,157424,V17
16,157424,V18
17,157424,V19


### Carry-Fill Patient-Constant Fields

`GENDER` and `Group` don't change over time for a given patient, so any grid rows added by the outer merge (visits with no original demographic row) get their values back by taking the per-patient mean — which is just the single existing value, since these fields are constant within a patient.

In [79]:
data = pd.merge(subgroup_meta, events_df, on=['PATNO', 'EVENT_ID'], how='outer')

data[['GENDER', 'Group']] = data[['GENDER', 'Group', 'PATNO']].groupby('PATNO').transform(
    lambda x: x.fillna(x.mean())
)

### Fill in Age for the Newly-Added Visit Rows

`AGE_AT_VISIT` does change across visits, so it can't be carry-filled the same way as `GENDER`/`Group`. Instead, each missing age is reconstructed by stepping 0.5 years (the visit spacing) forward from the last known age in row order.

In [80]:
def custom_increment_fill(group):
    """Forward-fill missing ages by incrementing 0.5 years per step.

    Walks a Series in order; whenever a value is missing, it's set to the
    previous (filled) value plus 0.5 (one visit's worth of elapsed time,
    given the ~6-month visit spacing used in this cohort). Non-missing
    values pass through unchanged and become the new "last value".

    Note: this is applied below via `groupby('Group')`, i.e. per
    subgroup rather than per patient. Within a subgroup, rows are
    processed in whatever order they currently appear in `data`, so a
    missing age is filled relative to the previous row in the *group*,
    not necessarily the same patient's own previous visit.
    """
    last_value = None  # last non-null value seen so far
    for i in range(len(group)):
        if pd.isna(group.iloc[i]):
            if last_value is not None:
                group.iloc[i] = last_value + 0.5
            last_value = group.iloc[i]
        else:
            last_value = group.iloc[i]
    return group

In [81]:
# Apply the increment-fill per subgroup (see note in the docstring above
# about this being grouped by 'Group' rather than by 'PATNO').
data['AGE_AT_VISIT'] = data.groupby('Group')['AGE_AT_VISIT'].transform(custom_increment_fill)

### Relabel Subgroup Codes

Same swap as in the MDS-UPDRS prep notebook: codes 0 and 2 are swapped (via a temporary `'C'` placeholder so the substitutions don't clobber each other). Net effect: original 0 -> 2, original 1 -> 1, original 2 -> 0. (Unlike the MDS-UPDRS notebook, this stops at the swapped numeric codes and does not relabel further to letters.)

In [82]:
data['Group'] = data['Group'].replace(0, 'C')   # temp label to free up 0
data['Group'] = data['Group'].replace(2, 0)     # 2 -> 0
data['Group'] = data['Group'].replace('C', 2)   # temp -> 2 (completes the 0/2 swap)

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_52827/187553215.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Group'] = data['Group'].replace('C', 2)   # temp -> 2 (completes the 0/2 swap)


### Export

In [83]:
data[['PATNO', 'Group', 'AGE_AT_VISIT', 'EVENT_ID', 'GENDER']] \
    .drop_duplicates() \
    .reset_index(drop=True) \
    .to_csv('./../../data/02_processed/PPMI/subgroups.csv')

In [84]:
data

,PATNO,AGE_AT_VISIT,EVENT_ID,GENDER,Group,agediag
0,3001,65.1,BL,1.0,2.0,64.260274
1,3001,65.6,V02,1.0,2.0,64.260274
2,3001,66.2,V04,1.0,2.0,64.260274
3,3001,66.7,V05,1.0,2.0,NaN
4,3001,67.3,V06,1.0,2.0,64.260274
...,...,...,...,...,...,...
10179,157424,81.7,V16,1.0,1.0,NaN
10180,157424,82.2,V17,1.0,1.0,NaN
10181,157424,82.7,V18,1.0,1.0,NaN
10182,157424,83.2,V19,1.0,1.0,NaN
